In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import zipfile

def gait_entropy_image(sequence_images):
    """
    Compute Gait Entropy Image.
    """
    # Convert sequence to numpy array
    seq_array = np.array(sequence_images)
    seq_len = len(sequence_images)
    
    # Calculate cumulative image
    cum_img = np.sum(seq_array, axis=0)
    cum_img = cum_img / 255  # Binarize
    
    # Calculate probabilities
    P1 = cum_img / seq_len              # p_1(x,y); probability pixel is 1
    P0 = (seq_len - cum_img) / seq_len  # p_0(x,y); probability pixel is 0
    
    # Calculate entropy
    H = -np.nan_to_num(P0 * np.log2(P0)) - np.nan_to_num(P1 * np.log2(P1))
    
    # Normalize to 0-255 range
    H_min, H_max = np.min(H), np.max(H)
    if H_max > H_min:
        geni = ((H - H_min) * 255 / (H_max - H_min)).astype(np.uint8)
    else:
        geni = np.zeros_like(H, dtype=np.uint8)
    
    return geni

def create_dataset(base_folder, output_folder, num_subjects=None, camera_angles=None):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    geni_images_folder = os.path.join(output_folder, 'geni_images')
    if not os.path.exists(geni_images_folder):
        os.makedirs(geni_images_folder)
    
    metadata = []
    counter = 0
    
    subject_folders = [f for f in os.listdir(base_folder) if f.isdigit()]
    if num_subjects is not None:
        subject_folders = subject_folders[:num_subjects]
    
    for subject_folder in subject_folders:
        counter += 1
        subject_path = os.path.join(base_folder, subject_folder, subject_folder)
        if not os.path.isdir(subject_path):
            print(f"No inner subject folder found for: {subject_folder}")
            continue
        
        for nm_folder in ['nm-01', 'nm-02', 'nm-03', 'nm-04', 'nm-05', 'nm-06']:
            nm_folder_path = os.path.join(subject_path, nm_folder)
            if not os.path.exists(nm_folder_path):
                print(f"No {nm_folder} folder found for subject: {subject_folder}")
                continue

            for cam_folder in camera_angles:
                cam_path = os.path.join(nm_folder_path, cam_folder)
                if not os.path.isdir(cam_path):
                    continue
                
                image_files = sorted([f for f in os.listdir(cam_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
                
                if not image_files:
                    print(f"No images found in {cam_path}")
                    continue
                
                sequence_images = []
                for file in image_files:
                    image_path = os.path.join(cam_path, file)
                    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
                    if image is not None and image.size > 0:
                        sequence_images.append(image)
                    else:
                        print(f"Failed to load image or empty image: {image_path}")
                
                if sequence_images:
                    geni = gait_entropy_image(sequence_images)
                    
                    if geni is not None and geni.size > 0:
                        geni_filename = f"{subject_folder}_{nm_folder}_{cam_folder}.png"
                        geni_path = os.path.join(geni_images_folder, geni_filename)
                        cv2.imwrite(geni_path, geni)
                        
                        metadata.append({
                            'image_filename': geni_filename,
                            'label': int(subject_folder),
                            'cam_angle': int(cam_folder),
                            'nm_sequence': nm_folder
                        })
                    else:
                        print(f"Failed to create valid GEnI for {cam_path}")
                
        print(f"{counter}: Finished processing subject: {subject_folder}")
    
    if metadata:
        df = pd.DataFrame(metadata)
        csv_path = os.path.join(output_folder, 'geni_metadata.csv')
        df.to_csv(csv_path, index=False)
        print(f"Successfully created {len(df)} GEnI images for {len(df['label'].unique())} subjects.")
        print(f"Metadata saved to: {csv_path}")
    else:
        print("No GEnI images were successfully created.")

def zip_dataset(output_folder, zip_filename):
    print(f"Creating zip file: {zip_filename}")
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(output_folder):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, output_folder)
                zipf.write(file_path, arcname)
    print(f"Zip file created: {zip_filename}")

# Example usage
if __name__ == "__main__":
    base_folder = '/kaggle/input/casia-b/GaitDatasetB-silh'
    output_folder = '/kaggle/working/geni_dataset'
    zip_filename = '/kaggle/working/geni_processed_dataset.zip'
    camera_angles = ['000', '018', '036', '054', '072', '090', '108', '126', '144', '162', '180']

    create_dataset(base_folder, output_folder, num_subjects=None, camera_angles=camera_angles)
    zip_dataset(output_folder, zip_filename)
    print(f"GEnI dataset has been created and zipped. You can now download {zip_filename} from the Kaggle output.")